# 04 - Agent Intelligence

## CyberForge AI - Agentic AI Capabilities

This notebook implements intelligent agent decision-making for:
- Task prioritization and execution planning
- Confidence-based decision scoring
- Gemini API integration for reasoning
- Autonomous threat response workflows

### Alignment:
- Integrates with desktop app agentic system
- Supports backend task queue management
- Provides explainable outputs for user transparency

In [ ]:
import json
import os
import time
import numpy as np
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from dataclasses import dataclass, asdict
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Load configuration
config_path = Path("../notebook_config.json")
with open(config_path) as f:
    CONFIG = json.load(f)

MODELS_DIR = Path(CONFIG["datasets_dir"]).parent / "models"
AGENT_DIR = MODELS_DIR.parent / "agent"
AGENT_DIR.mkdir(exist_ok=True)

print(f"✓ Configuration loaded")
print(f"✓ Agent output: {AGENT_DIR}")

## 1. Agent Task Definitions

In [ ]:
class TaskPriority(Enum):
    CRITICAL = 4
    HIGH = 3
    MEDIUM = 2
    LOW = 1
    BACKGROUND = 0

@dataclass
class AgentTask:
    """Represents a task the agent can execute"""
    task_id: str
    task_type: str
    priority: int
    target: str
    context: Dict
    created_at: float
    confidence: float = 0.0
    status: str = "pending"
    result: Dict = None
    
    def to_dict(self) -> Dict:
        return asdict(self)

@dataclass
class AgentDecision:
    """Represents an agent decision with reasoning"""
    action: str
    confidence: float
    reasoning: str
    evidence: List[str]
    risk_level: str
    recommended_follow_up: List[str]
    
    def to_dict(self) -> Dict:
        return asdict(self)

print("✓ Task definitions loaded")

## 2. Decision Scoring Engine

In [ ]:
class DecisionScoringEngine:
    """
    Calculates confidence scores for agent decisions.
    Combines model predictions with heuristic rules.
    """
    
    # Threat severity weights
    SEVERITY_WEIGHTS = {
        'critical': 1.0,
        'high': 0.8,
        'medium': 0.5,
        'low': 0.3,
        'info': 0.1
    }
    
    # Evidence type weights
    EVIDENCE_WEIGHTS = {
        'model_prediction': 0.4,
        'signature_match': 0.3,
        'behavioral_pattern': 0.2,
        'heuristic_rule': 0.1
    }
    
    def __init__(self, confidence_threshold: float = 0.7):
        self.confidence_threshold = confidence_threshold
    
    def calculate_threat_score(self, indicators: List[Dict]) -> Tuple[float, str]:
        """Calculate threat score from multiple indicators"""
        if not indicators:
            return 0.0, 'low'
        
        # Weighted average of indicator scores
        total_weight = 0
        total_score = 0
        
        for indicator in indicators:
            severity = indicator.get('severity', 'low')
            confidence = indicator.get('confidence', 0.5)
            evidence_type = indicator.get('evidence_type', 'heuristic_rule')
            
            weight = self.SEVERITY_WEIGHTS.get(severity, 0.3) * \
                     self.EVIDENCE_WEIGHTS.get(evidence_type, 0.1)
            
            total_weight += weight
            total_score += confidence * weight
        
        if total_weight == 0:
            return 0.0, 'low'
        
        final_score = total_score / total_weight
        
        # Determine risk level
        if final_score >= 0.8:
            risk = 'critical'
        elif final_score >= 0.6:
            risk = 'high'
        elif final_score >= 0.4:
            risk = 'medium'
        elif final_score >= 0.2:
            risk = 'low'
        else:
            risk = 'info'
        
        return final_score, risk
    
    def should_act(self, score: float, task_type: str) -> bool:
        """Determine if agent should act based on score"""
        # Different thresholds for different task types
        thresholds = {
            'block_threat': 0.85,
            'quarantine': 0.75,
            'alert': 0.5,
            'scan': 0.3,
            'monitor': 0.1
        }
        
        threshold = thresholds.get(task_type, self.confidence_threshold)
        return score >= threshold
    
    def prioritize_tasks(self, tasks: List[AgentTask]) -> List[AgentTask]:
        """Sort tasks by priority and confidence"""
        return sorted(tasks, 
                      key=lambda t: (t.priority, t.confidence), 
                      reverse=True)

scoring_engine = DecisionScoringEngine()
print("✓ Decision Scoring Engine initialized")

## 3. Gemini API Integration for Reasoning

In [ ]:
try:
    import google.generativeai as genai
    GEMINI_AVAILABLE = True
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'google-generativeai', '-q'])
    import google.generativeai as genai
    GEMINI_AVAILABLE = True

class GeminiReasoningEngine:
    """
    Uses Gemini API for intelligent threat reasoning.
    Provides explainable AI outputs.
    """
    
    SYSTEM_PROMPT = """You are a cybersecurity AI agent analyzing security threats.
Your role is to:
1. Analyze security indicators and threat patterns
2. Provide clear, actionable recommendations
3. Explain your reasoning in a way users can understand
4. Prioritize user safety while minimizing false positives

Always respond with JSON containing:
- action: recommended action (block, alert, monitor, allow)
- confidence: 0.0-1.0 confidence score
- reasoning: brief explanation
- evidence: list of supporting evidence
- risk_level: critical/high/medium/low/info
- recommended_follow_up: list of next steps"""
    
    def __init__(self):
        self.api_key = CONFIG.get('gemini_api_key', os.environ.get('GEMINI_API_KEY'))
        self.model = None
        
        if self.api_key:
            try:
                genai.configure(api_key=self.api_key)
                self.model = genai.GenerativeModel('gemini-2.0-flash')
                print("  ✓ Gemini API connected")
            except Exception as e:
                print(f"  ⚠ Gemini API error: {e}")
        else:
            print("  ⚠ No Gemini API key found (will use fallback reasoning)")
    
    def analyze_threat(self, threat_data: Dict) -> AgentDecision:
        """Analyze threat and generate decision with reasoning"""
        if self.model:
            return self._gemini_analyze(threat_data)
        else:
            return self._fallback_analyze(threat_data)
    
    def _gemini_analyze(self, threat_data: Dict) -> AgentDecision:
        """Use Gemini for threat analysis"""
        prompt = f"""{self.SYSTEM_PROMPT}

Analyze this security threat:
{json.dumps(threat_data, indent=2)}

Provide your analysis as JSON."""
        
        try:
            response = self.model.generate_content(prompt)
            
            # Parse response
            text = response.text
            # Extract JSON from response
            if '```json' in text:
                text = text.split('```json')[1].split('```')[0]
            elif '```' in text:
                text = text.split('```')[1].split('```')[0]
            
            result = json.loads(text)
            
            return AgentDecision(
                action=result.get('action', 'monitor'),
                confidence=float(result.get('confidence', 0.5)),
                reasoning=result.get('reasoning', 'Analysis pending'),
                evidence=result.get('evidence', []),
                risk_level=result.get('risk_level', 'medium'),
                recommended_follow_up=result.get('recommended_follow_up', [])
            )
        except Exception as e:
            print(f"  ⚠ Gemini error: {e}")
            return self._fallback_analyze(threat_data)
    
    def _fallback_analyze(self, threat_data: Dict) -> AgentDecision:
        """Rule-based fallback when Gemini unavailable"""
        risk_score = threat_data.get('risk_score', 0.5)
        indicators = threat_data.get('indicators', [])
        
        # Determine action based on risk score
        if risk_score >= 0.8:
            action = 'block'
            risk_level = 'critical'
        elif risk_score >= 0.6:
            action = 'alert'
            risk_level = 'high'
        elif risk_score >= 0.4:
            action = 'monitor'
            risk_level = 'medium'
        else:
            action = 'allow'
            risk_level = 'low'
        
        return AgentDecision(
            action=action,
            confidence=risk_score,
            reasoning=f"Risk score: {risk_score:.2f}. Indicators found: {len(indicators)}",
            evidence=[str(i) for i in indicators[:3]],
            risk_level=risk_level,
            recommended_follow_up=['Continue monitoring', 'Review threat logs']
        )

reasoning_engine = GeminiReasoningEngine()
print("✓ Gemini Reasoning Engine initialized")

## 4. Task Queue Manager

In [ ]:
import uuid
from collections import deque

class AgentTaskQueue:
    """
    Manages agent task queue for autonomous operation.
    Supports priority-based execution and task lifecycle.
    """
    
    def __init__(self, max_concurrent: int = 5):
        self.pending = deque()
        self.active = []
        self.completed = []
        self.failed = []
        self.max_concurrent = max_concurrent
    
    def add_task(self, task_type: str, target: str, context: Dict,
                 priority: TaskPriority = TaskPriority.MEDIUM) -> AgentTask:
        """Add a new task to the queue"""
        task = AgentTask(
            task_id=str(uuid.uuid4())[:8],
            task_type=task_type,
            priority=priority.value,
            target=target,
            context=context,
            created_at=time.time()
        )
        
        self.pending.append(task)
        self._reorder_queue()
        
        return task
    
    def _reorder_queue(self):
        """Reorder queue by priority"""
        self.pending = deque(sorted(self.pending, 
                                    key=lambda t: (t.priority, -t.created_at),
                                    reverse=True))
    
    def get_next_task(self) -> Optional[AgentTask]:
        """Get next task to execute"""
        if len(self.active) >= self.max_concurrent:
            return None
        
        if self.pending:
            task = self.pending.popleft()
            task.status = 'active'
            self.active.append(task)
            return task
        
        return None
    
    def complete_task(self, task_id: str, result: Dict, success: bool = True):
        """Mark task as completed"""
        for i, task in enumerate(self.active):
            if task.task_id == task_id:
                task.status = 'completed' if success else 'failed'
                task.result = result
                
                if success:
                    self.completed.append(task)
                else:
                    self.failed.append(task)
                
                del self.active[i]
                return
    
    def get_stats(self) -> Dict:
        """Get queue statistics"""
        return {
            'pending': len(self.pending),
            'active': len(self.active),
            'completed': len(self.completed),
            'failed': len(self.failed),
            'total': len(self.pending) + len(self.active) + len(self.completed) + len(self.failed)
        }

task_queue = AgentTaskQueue()
print("✓ Task Queue Manager initialized")

## 5. Autonomous Agent Orchestrator

In [ ]:
class CyberForgeAgent:
    """
    Main autonomous agent for CyberForge.
    Orchestrates threat detection, analysis, and response.
    """
    
    def __init__(self):
        self.scoring_engine = DecisionScoringEngine()
        self.reasoning_engine = GeminiReasoningEngine()
        self.task_queue = AgentTaskQueue()
        self.action_history = []
    
    def analyze_website(self, url: str, scraped_data: Dict) -> AgentDecision:
        """Analyze a website for threats"""
        # Extract indicators
        indicators = self._extract_indicators(scraped_data)
        
        # Calculate threat score
        score, risk_level = self.scoring_engine.calculate_threat_score(indicators)
        
        # Get AI reasoning
        threat_data = {
            'url': url,
            'risk_score': score,
            'indicators': indicators,
            'security_report': scraped_data.get('security_report', {})
        }
        
        decision = self.reasoning_engine.analyze_threat(threat_data)
        
        # Log decision
        self._log_action(url, decision)
        
        return decision
    
    def _extract_indicators(self, data: Dict) -> List[Dict]:
        """Extract threat indicators from scraped data"""
        indicators = []
        
        # Check security report
        security = data.get('security_report', {})
        if not security.get('is_https', True):
            indicators.append({
                'type': 'insecure_protocol',
                'severity': 'medium',
                'confidence': 0.9,
                'evidence_type': 'signature_match'
            })
        
        if security.get('mixed_content', False):
            indicators.append({
                'type': 'mixed_content',
                'severity': 'medium',
                'confidence': 0.85,
                'evidence_type': 'signature_match'
            })
        
        # Check console errors
        console_logs = data.get('console_logs', [])
        errors = [log for log in console_logs if log.get('level') == 'error']
        if len(errors) > 5:
            indicators.append({
                'type': 'excessive_errors',
                'severity': 'low',
                'confidence': 0.6,
                'evidence_type': 'behavioral_pattern'
            })
        
        # Check for suspicious network requests
        requests = data.get('network_requests', [])
        suspicious_domains = ['malware', 'phishing', 'hack', 'tracker']
        for req in requests:
            url = req.get('url', '').lower()
            if any(s in url for s in suspicious_domains):
                indicators.append({
                    'type': 'suspicious_request',
                    'severity': 'high',
                    'confidence': 0.75,
                    'evidence_type': 'signature_match',
                    'details': url[:100]
                })
        
        return indicators
    
    def _log_action(self, target: str, decision: AgentDecision):
        """Log agent action for audit trail"""
        self.action_history.append({
            'timestamp': time.time(),
            'target': target,
            'action': decision.action,
            'confidence': decision.confidence,
            'risk_level': decision.risk_level
        })
    
    def get_action_summary(self) -> Dict:
        """Get summary of agent actions"""
        if not self.action_history:
            return {'total_actions': 0}
        
        actions = [a['action'] for a in self.action_history]
        return {
            'total_actions': len(self.action_history),
            'action_counts': {a: actions.count(a) for a in set(actions)},
            'avg_confidence': np.mean([a['confidence'] for a in self.action_history])
        }

agent = CyberForgeAgent()
print("✓ CyberForge Agent initialized")

## 6. Test Agent Decision Making

In [ ]:
# Test with sample threat data
test_data = {
    'url': 'https://suspicious-login.example.com/verify',
    'security_report': {
        'is_https': True,
        'mixed_content': True,
        'insecure_cookies': True
    },
    'console_logs': [
        {'level': 'error', 'message': 'CORS policy violation'},
        {'level': 'error', 'message': 'Failed to load resource'},
    ],
    'network_requests': [
        {'url': 'https://tracker.malicious.com/collect', 'type': 'xhr'},
        {'url': 'https://cdn.example.com/app.js', 'type': 'script'}
    ]
}

print("Testing agent analysis...\n")
decision = agent.analyze_website(test_data['url'], test_data)

print(f"Decision: {decision.action}")
print(f"Confidence: {decision.confidence:.2%}")
print(f"Risk Level: {decision.risk_level}")
print(f"\nReasoning: {decision.reasoning}")
print(f"\nEvidence:")
for e in decision.evidence[:3]:
    print(f"  - {e}")
print(f"\nRecommended Follow-up:")
for r in decision.recommended_follow_up[:3]:
    print(f"  - {r}")

## 7. Save Agent Configuration

In [ ]:
# Save agent configuration and modules
agent_config = {
    'version': '1.0.0',
    'confidence_threshold': 0.7,
    'max_concurrent_tasks': 5,
    'severity_weights': DecisionScoringEngine.SEVERITY_WEIGHTS,
    'evidence_weights': DecisionScoringEngine.EVIDENCE_WEIGHTS,
    'task_priorities': {p.name: p.value for p in TaskPriority},
    'gemini_model': 'gemini-2.0-flash'
}

config_path = AGENT_DIR / "agent_config.json"
with open(config_path, 'w') as f:
    json.dump(agent_config, f, indent=2)

print(f"✓ Agent config saved to: {config_path}")

In [ ]:
# Save agent module for backend import
agent_module = '''
"""CyberForge Agent Intelligence Module"""

import json
import time
import numpy as np
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Any, Optional

@dataclass
class AgentDecision:
    action: str
    confidence: float
    reasoning: str
    evidence: List[str]
    risk_level: str
    recommended_follow_up: List[str]
    
    def to_dict(self):
        return asdict(self)

class DecisionEngine:
    SEVERITY_WEIGHTS = {"critical": 1.0, "high": 0.8, "medium": 0.5, "low": 0.3, "info": 0.1}
    
    def calculate_threat_score(self, indicators: List[Dict]) -> tuple:
        if not indicators:
            return 0.0, "low"
        scores = [i.get("confidence", 0.5) * self.SEVERITY_WEIGHTS.get(i.get("severity", "low"), 0.3) 
                  for i in indicators]
        score = sum(scores) / len(scores) if scores else 0
        risk = "critical" if score >= 0.8 else "high" if score >= 0.6 else "medium" if score >= 0.4 else "low"
        return score, risk

class CyberForgeAgent:
    def __init__(self):
        self.engine = DecisionEngine()
    
    def analyze(self, url: str, data: Dict) -> Dict:
        indicators = self._extract_indicators(data)
        score, risk = self.engine.calculate_threat_score(indicators)
        action = "block" if score >= 0.8 else "alert" if score >= 0.6 else "monitor" if score >= 0.4 else "allow"
        
        return AgentDecision(
            action=action,
            confidence=score,
            reasoning=f"Threat score: {score:.2f}. {len(indicators)} indicators found.",
            evidence=[str(i) for i in indicators[:3]],
            risk_level=risk,
            recommended_follow_up=["Continue monitoring"]
        ).to_dict()
    
    def _extract_indicators(self, data: Dict) -> List[Dict]:
        indicators = []
        sec = data.get("security_report", {})
        if not sec.get("is_https", True):
            indicators.append({"type": "insecure", "severity": "medium", "confidence": 0.9})
        if sec.get("mixed_content"):
            indicators.append({"type": "mixed_content", "severity": "medium", "confidence": 0.85})
        return indicators
'''

module_path = AGENT_DIR / "cyberforge_agent.py"
with open(module_path, 'w') as f:
    f.write(agent_module)

print(f"✓ Agent module saved to: {module_path}")

## 8. Summary

In [ ]:
print("\n" + "=" * 60)
print("AGENT INTELLIGENCE COMPLETE")
print("=" * 60)

print(f"""
🤖 Agent Capabilities:
   - Decision Scoring: Weighted threat assessment
   - Gemini Integration: AI-powered reasoning
   - Task Queue: Priority-based execution
   - Action History: Full audit trail

📊 Test Results:
   - Action: {decision.action}
   - Confidence: {decision.confidence:.2%}
   - Risk Level: {decision.risk_level}

📁 Output Files:
   - Config: {AGENT_DIR}/agent_config.json
   - Module: {AGENT_DIR}/cyberforge_agent.py

Next step:
  → 05_model_validation.ipynb
""")
print("=" * 60)